# CIC-IDS2017 Colab Practice Notebook

이 노트북은 실습용 언더샘플링 데이터셋을 Google Colab에서 불러와 전처리, 모델 실험, 저장/불러오기, 테스트 평가까지 한 번에 진행합니다.

In [ ]:
!pip -q install xgboost torch scikit-learn joblib pandas numpy

In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# TODO: 본인 Drive 경로에 맞게 수정하세요.
PROJECT_ROOT = Path('/content/drive/MyDrive/cic')
DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACT_DIR = PROJECT_ROOT / 'outputs'
MODEL_DIR = ARTIFACT_DIR / 'models'
REPORT_DIR = ARTIFACT_DIR / 'reports'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_DIR =', DATA_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)

## 1. Load Practice CSV

- `cic2017_practice_train.csv`: 실습용 개발 데이터
- `cic2017_practice_test.csv`: 최종 테스트 데이터
- 노트북 안에서는 `train -> train/valid = 8:2 random split` 을 다시 수행합니다.

In [ ]:
import json
import random

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv(DATA_DIR / 'cic2017_train.csv')
test_df = pd.read_csv(DATA_DIR / 'cic2017_test.csv')

print('train_df shape:', train_df.shape)
print('test_df shape:', test_df.shape)
display(train_df.head())

In [ ]:
LABEL_COL = 'Label'

train_df, valid_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df[LABEL_COL],
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('train split:', train_df.shape)
print('valid split:', valid_df.shape)
print('test split :', test_df.shape)
display(train_df[LABEL_COL].value_counts().rename('count').to_frame().head(10))

## 2. Preprocess Like `code_mat.py`

핵심 흐름은 다음과 같습니다.

- 라벨 문자열 정리
- 비수치형 컬럼 frequency encoding
- 수치형 컬럼 NaN/inf 처리
- train 기준으로 스케일링
- label encoding

In [ ]:
LABEL_REPLACEMENTS = {
    'web-attack-�-brute-force': 'web-attack-brute-force',
    'web-attack-�-xss': 'web-attack-xss',
    'web-attack-�-sql-injection': 'web-attack-sql-injection',
    'web-attack-brute force': 'web-attack-brute-force',
    'web-attack-sql injection': 'web-attack-sql-injection',
}


def normalize_labels(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(' ', '-', regex=False)
        .str.replace('–', '-', regex=False)
        .replace({'normal': 'benign'})
        .replace(LABEL_REPLACEMENTS)
    )


def preprocess_splits(train_df, valid_df, test_df, label_col='Label'):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    for frame in [train_df, valid_df, test_df]:
        frame[label_col] = normalize_labels(frame[label_col])

    feature_cols = [c for c in train_df.columns if c != label_col]
    cat_cols = list(train_df[feature_cols].select_dtypes(include=['object', 'category']).columns)
    num_cols = [c for c in feature_cols if c not in cat_cols]

    for col in cat_cols:
        freq_map = train_df[col].astype(str).value_counts(normalize=True)
        for frame in [train_df, valid_df, test_df]:
            frame[col] = frame[col].astype(str).map(freq_map).fillna(0.0).astype('float32')

    for col in num_cols:
        for frame in [train_df, valid_df, test_df]:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')

    for frame in [train_df, valid_df, test_df]:
        frame[feature_cols] = frame[feature_cols].replace([np.inf, -np.inf], np.nan)
        frame[feature_cols] = frame[feature_cols].fillna(0.0)

    var_ser = train_df[feature_cols].var(numeric_only=True)
    zero_var_cols = var_ser[var_ser == 0].index.tolist()
    if zero_var_cols:
        feature_cols = [c for c in feature_cols if c not in zero_var_cols]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[feature_cols]).astype('float32')
    X_valid = scaler.transform(valid_df[feature_cols]).astype('float32')
    X_test = scaler.transform(test_df[feature_cols]).astype('float32')

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(train_df[label_col])
    y_valid = label_encoder.transform(valid_df[label_col])
    y_test = label_encoder.transform(test_df[label_col])

    artifacts = {
        'feature_cols': feature_cols,
        'cat_cols': cat_cols,
        'scaler': scaler,
        'label_encoder': label_encoder,
    }
    return X_train, y_train, X_valid, y_valid, X_test, y_test, artifacts


X_train, y_train, X_valid, y_valid, X_test, y_test, artifacts = preprocess_splits(train_df, valid_df, test_df, LABEL_COL)
joblib.dump(artifacts, MODEL_DIR / 'preprocess_artifacts.joblib')

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test :', X_test.shape)
print('classes:', artifacts['label_encoder'].classes_)

In [ ]:
def save_report(y_true, y_pred, label_encoder, save_path):
    labels = list(range(len(label_encoder.classes_)))
    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for class_name in label_encoder.classes_:
        rows.append({
            'class': class_name,
            'precision': report[class_name]['precision'],
            'recall': report[class_name]['recall'],
            'f1': report[class_name]['f1-score'],
            'support': report[class_name]['support'],
        })
    rows.append({
        'class': 'macro_avg',
        'precision': report['macro avg']['precision'],
        'recall': report['macro avg']['recall'],
        'f1': report['macro avg']['f1-score'],
        'support': report['macro avg']['support'],
    })
    df = pd.DataFrame(rows)
    df.to_csv(save_path, index=False)
    print(f'Saved metrics: {save_path}')
    print(df.to_string(index=False))
    return df


def save_sklearn_like_model(model, model_name, suffix='joblib'):
    save_path = MODEL_DIR / f'{model_name}.{suffix}'
    joblib.dump(model, save_path)
    print(f'Saved model: {save_path}')
    return save_path


def save_torch_model(state, model_name):
    save_path = MODEL_DIR / f'{model_name}.pth'
    torch.save(state, save_path)
    print(f'Saved model: {save_path}')
    return save_path


def evaluate_predictions(model_name, split_name, y_true, y_pred, label_encoder):
    save_path = REPORT_DIR / f'{model_name}_{split_name}_metrics.csv'
    return save_report(y_true, y_pred, label_encoder, save_path)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

## 3. Student Model Area (TBD)

아래 구간은 학생 실습용으로 비워 두세요. 선택지 예시는 다음 5가지입니다.

1. Logistic Regression
2. Random Forest
3. XGBoost
4. MLP (Fully Connected Neural Network)
5. LSTM

In [ ]:
# TODO(student): 아래 형식을 유지해서 모델을 구현해 보세요.
# 예시 아이디어
# 1) sklearn LogisticRegression
# 2) RandomForestClassifier
# 3) XGBClassifier
# 4) PyTorch MLP
# 5) PyTorch LSTM

STUDENT_MODEL_NAME = 'student_baseline'

# sklearn 스타일 예시 템플릿
# from sklearn.linear_model import LogisticRegression
# student_model = LogisticRegression(max_iter=300, n_jobs=-1)
# student_model.fit(X_train, y_train)
# save_sklearn_like_model(student_model, STUDENT_MODEL_NAME)
# valid_pred = student_model.predict(X_valid)
# test_pred = student_model.predict(X_test)
# student_valid_report = evaluate_predictions(STUDENT_MODEL_NAME, 'valid', y_valid, valid_pred, artifacts['label_encoder'])
# student_test_report = evaluate_predictions(STUDENT_MODEL_NAME, 'test', y_test, test_pred, artifacts['label_encoder'])
# display(student_valid_report)
# display(student_test_report)

# PyTorch 스타일 예시 템플릿
# student_state = {
#     'state_dict': student_model.state_dict(),
#     'label_classes': artifacts['label_encoder'].classes_.tolist(),
#     'feature_cols': artifacts['feature_cols'],
# }
# save_torch_model(student_state, STUDENT_MODEL_NAME)


## 4. Demo 1: XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=SEED,
    early_stopping_rounds=20,
)
xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=10,
)

xgb_valid_pred = xgb_model.predict(X_valid)
xgb_test_pred = xgb_model.predict(X_test)

xgb_model.save_model(MODEL_DIR / 'xgb_demo.json')
save_sklearn_like_model(xgb_model, 'xgb_demo')

xgb_valid_report = evaluate_predictions('xgb_demo', 'valid', y_valid, xgb_valid_pred, artifacts['label_encoder'])
xgb_test_report = evaluate_predictions('xgb_demo', 'test', y_test, xgb_test_pred, artifacts['label_encoder'])

In [ ]:
loaded_xgb = XGBClassifier()
loaded_xgb.load_model(MODEL_DIR / 'xgb_demo.json')
loaded_xgb_test_pred = loaded_xgb.predict(X_test)
loaded_xgb_test_report = save_report(
    y_test,
    loaded_xgb_test_pred,
    artifacts['label_encoder'],
    REPORT_DIR / 'xgb_test_metrics_reloaded.csv',
)

## 5. Demo 2: LSTM

탭형 데이터를 간단히 LSTM에 넣기 위해 `[batch, seq_len=num_features, input_size=1]` 형태로 reshape 합니다.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def make_lstm_loader(X, y, batch_size=256, shuffle=False):
    X_seq = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y_tensor = torch.tensor(y, dtype=torch.long)
    ds = TensorDataset(X_seq, y_tensor)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


class TabularLSTM(nn.Module):
    def __init__(self, hidden_size, num_classes, num_layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        output, _ = self.lstm(x)
        last_hidden = output[:, -1, :]
        return self.classifier(last_hidden)


train_loader = make_lstm_loader(X_train, y_train, shuffle=True)
valid_loader = make_lstm_loader(X_valid, y_valid)
test_loader = make_lstm_loader(X_test, y_test)


In [ ]:
def evaluate_lstm(model, data_loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            logits = model(batch_x)
            pred = logits.argmax(dim=1).cpu().numpy()
            preds.append(pred)
            targets.append(batch_y.numpy())
    return np.concatenate(targets), np.concatenate(preds)


lstm_model = TabularLSTM(hidden_size=64, num_classes=len(artifacts['label_encoder'].classes_)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)
num_epochs = 5

for epoch in range(num_epochs):
    lstm_model.train()
    running_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        logits = lstm_model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)

    valid_true, valid_pred = evaluate_lstm(lstm_model, valid_loader, device)
    valid_report = classification_report(valid_true, valid_pred, output_dict=True, zero_division=0)
    avg_loss = running_loss / len(train_loader.dataset)
    print(f'Epoch {epoch + 1}/{num_epochs} | loss={avg_loss:.4f} | valid_macro_f1={valid_report["macro avg"]["f1-score"]:.4f}')

lstm_ckpt = {
    'state_dict': lstm_model.state_dict(),
    'label_classes': artifacts['label_encoder'].classes_.tolist(),
    'feature_cols': artifacts['feature_cols'],
}
save_torch_model(lstm_ckpt, 'lstm_demo')

lstm_valid_true, lstm_valid_pred = evaluate_lstm(lstm_model, valid_loader, device)
lstm_test_true, lstm_test_pred = evaluate_lstm(lstm_model, test_loader, device)

lstm_valid_report = evaluate_predictions('lstm_demo', 'valid', lstm_valid_true, lstm_valid_pred, artifacts['label_encoder'])
lstm_test_report = evaluate_predictions('lstm_demo', 'test', lstm_test_true, lstm_test_pred, artifacts['label_encoder'])

In [ ]:
loaded_ckpt = torch.load(MODEL_DIR / 'lstm_demo.pth', map_location=device)
loaded_lstm = TabularLSTM(hidden_size=64, num_classes=len(loaded_ckpt['label_classes'])).to(device)
loaded_lstm.load_state_dict(loaded_ckpt['state_dict'])

reloaded_test_true, reloaded_test_pred = evaluate_lstm(loaded_lstm, test_loader, device)
reloaded_lstm_report = save_report(
    reloaded_test_true,
    reloaded_test_pred,
    artifacts['label_encoder'],
    REPORT_DIR / 'lstm_test_metrics_reloaded.csv',
)

In [ ]:
summary = {
    'model_dir': str(MODEL_DIR),
    'report_dir': str(REPORT_DIR),
    'saved_files': sorted([p.name for p in MODEL_DIR.glob('*')]) + sorted([p.name for p in REPORT_DIR.glob('*')]),
}
with open(ARTIFACT_DIR / 'run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))